# DSE 230 - UC San Diego
## SageMaker Classification Demo with XGBoost
### INFERENCE NOTEBOOK

## Startup

In [ ]:
import os, sagemaker
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sagemaker import get_execution_role

In [ ]:
# Define IAM role- this will be necessary when defining your model
iam_role = get_execution_role()

# Set SageMaker session handle
sess = sagemaker.Session()

# Set the region of the instance 
my_region = sess.boto_session.region_name

# Set the sagemaker client
sagemaker_client = sagemaker.Session().boto_session.client('sagemaker')

print("Success - the SageMaker instance is in the " + my_region + " region")

## Using Model for Inference
### NOTE - Inference can be done in a separate notebook/application
* For this exercise, we will extract the endpoint from a `Predictor` object
* In practice, the endpoint can be obtained from the SageMaker dashboard once the model is deployed

## Get deployed endpoints

#### We will use the SageMaker client to get the list of active endpoints

**NOTE:** `list_endpoints()` method returns a list of dictionaries.

In [ ]:
# List endpoints
endpoints = sagemaker_client.list_endpoints()

endpoints

In [ ]:
# Print endpoint information
for endpoint in endpoints['Endpoints']:
    print("Endpoint Name:", endpoint['EndpointName'])
    # print("EndpointArn:", endpoint['EndpointArn'])
    print("Status:", endpoint['EndpointStatus'])
    print()

## Real-Time Inference

### Use model endpoint to perform inference

Use endpoint name from above cell.


In [ ]:
predictor = sagemaker.predictor.Predictor(endpoint_name='<ENDPOINT_NAME>',
                                          sagemaker_session=sess,
                                          serializer=sagemaker.serializers.CSVSerializer(),
                                          deserializer=sagemaker.deserializers.BytesDeserializer())

### Prepare test data
Drop the label column and load values into an array

In [ ]:
# Set S3 bucket name and folder
# NOTE:  Enter your bucket name and folder here.

bucket = "<BUCKET>"
prefix = "<FOLDER>"
print('Using bucket ' + bucket)

data_fname = "s3://{}/{}/{}".format(bucket, prefix, "data/wine_test.csv")
test_df  = pd.read_csv(data_fname)
print('Reading test data from', data_fname)

In [ ]:
# test_df = pd.read_csv('test_data.csv')
print(test_df.shape)
test_df.head(1)

In [ ]:
test_df_array = test_df.drop(['Class'], axis=1).values

In [ ]:
test_df_array[0]

### Evaluate results of tuned model with real-time inference
* Predictions are returned as byte object, so the contents need to be decoded into string and converted to number array.
* Then performance metrics are calculated.

In [ ]:
predictions = predictor.predict(data=test_df_array).decode('utf-8') # predict!
predictions_array = np.fromstring(predictions, sep=',')  #and turn the prediction into an array

In [ ]:
from sklearn.metrics import accuracy_score

y_true = test_df['Class'].values
y_pred = predictions_array.astype(int)

print(y_pred)
print(y_true)

print("Accuracy of tuned model: %.3f" % accuracy_score(y_true,y_pred))